In [21]:
# loading in necessary libraries and cleaned data from last file
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('data/cleaned_survey.csv')
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4448 entries, 0 to 4447
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ResponseId              4448 non-null   int64  
 1   Age                     4448 non-null   str    
 2   EdLevel                 4448 non-null   str    
 3   Employment              4448 non-null   str    
 4   WorkExp                 4448 non-null   float64
 5   YearsCode               4448 non-null   float64
 6   DevType                 4448 non-null   str    
 7   OrgSize                 4448 non-null   str    
 8   ICorPM                  4448 non-null   str    
 9   RemoteWork              4448 non-null   str    
 10  Industry                4448 non-null   str    
 11  Country                 4448 non-null   str    
 12  LanguageHaveWorkedWith  4448 non-null   str    
 13  DatabaseHaveWorkedWith  3760 non-null   str    
 14  annual_salary_usd       4448 non-null   float64
dty

# ENCODING & MODELING
### (picks up where exploredata.ipynb left off, trying to avoid lengthiness)

## ENCODING 'Age', 'EdLevel', and 'OrgSize'

In [22]:
# idea is to ordinal encode, encoding over ordered integers (e.g. 1, 2, 3, 4, ...)
# different than one-hot, which creates binary indicator columns
df_clean['Age'].value_counts()

Age
25-34 years old      1686
35-44 years old      1426
45-54 years old       604
18-24 years old       439
55-64 years old       246
65 years or older      42
Prefer not to say       5
Name: count, dtype: int64

In [23]:
# prefer not to say has only 5 respondants
# deciding to drop those rows instead of forcing those responses into a numerical category
df_clean = df_clean[df_clean['Age'] != 'Prefer not to say']
print(f'Rows Remaining: {len(df_clean)}')

Rows Remaining: 4443


In [24]:
# ordinal encoding for ages
age_map = {
    '18-24 years old': 1,
    '25-34 years old': 2,
    '35-44 years old': 3,
    '45-54 years old': 4,
    '55-64 years old': 5,
    '65 years or older': 6
}
df_clean['Age_encoded'] = df_clean['Age'].map(age_map)

In [25]:
# also intending to ordinal encode 'EdLevel', but looking at values first
# number of respondants who put 'Other' feels like more substantial than the respondants for 'prefer not to say' for Age
# primary/elementary school feels concerning though, maybe they didn't complete and were self-taught?
df_clean['EdLevel'].value_counts()

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          2024
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       1321
Some college/university study without earning a degree                                 516
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         216
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     181
Associate degree (A.A., A.S., etc.)                                                    135
Other (please specify):                                                                 35
Primary/elementary school                                                               15
Name: count, dtype: int64

In [26]:
# think i'm going to drop respondants who said 'primary/elemantary school', lowest number of respondants as well
df_clean = df_clean[df_clean['EdLevel'] != 'Primary/elementary school']
print(f'Rows Remaining: {len(df_clean)}')

Rows Remaining: 4428


In [27]:
# ordinal encoding for education levels
edlevel_map = {
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 1,
    'Some college/university study without earning a degree': 2,
    'Associate degree (A.A., A.S., etc.)': 3,
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": 4,
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 6,
    'Other (please specify):': 0
}
df_clean['EdLevel_encoded'] = df_clean['EdLevel'].map(edlevel_map)

In [28]:
# intending to ordinal encode 'OrgSize' too, inspecting first
# all seem relevant and substantial, keeping all
df_clean['OrgSize'].value_counts()

OrgSize
20 to 99 employees                                    1316
100 to 499 employees                                   808
Less than 20 employees                                 643
10,000 or more employees                               597
1,000 to 4,999 employees                               522
500 to 999 employees                                   267
5,000 to 9,999 employees                               176
Just me - I am a freelancer, sole proprietor, etc.      99
Name: count, dtype: int64

In [29]:
# ordinal encoding for orgization size
orgsize_map = {
    'Just me - I am a freelancer, sole proprietor, etc.': 1,
    'Less than 20 employees': 2,
    '20 to 99 employees': 3,
    '100 to 499 employees': 4,
    '500 to 999 employees': 5,
    '1,000 to 4,999 employees': 6,
    '5,000 to 9,999 employees': 7,
    '10,000 or more employees': 8
}
df_clean['OrgSize_encoded'] = df_clean['OrgSize'].map(orgsize_map)

In [30]:
# checking to see if everything mapped over alright, should be no null values
print(df_clean[['Age_encoded', 'EdLevel_encoded', 'OrgSize_encoded']].isna().sum())

Age_encoded        0
EdLevel_encoded    0
OrgSize_encoded    0
dtype: int64


In [31]:
# for nominal/categorical columns, choosing to one-hot encode
# no hierarchy where one thing is "more" or "less" than another
# each will get its own category of whether it's "filled" or not
nominal_columns = ['DevType', 'Country', 'Industry', 'Employment', 'ICorPM', 'RemoteWork']

df_encoded = pd.get_dummies(df_clean, columns = nominal_columns, drop_first = True)

In [32]:
# need to multilabel encode for languges and databases used, since more than one instance can belong to a cell
# semicolon separated values
language_dummies = df_clean['LanguageHaveWorkedWith'].str.get_dummies(sep=';').add_prefix('lang_')
database_dummies = df_clean['DatabaseHaveWorkedWith'].str.get_dummies(sep=';').add_prefix('db_')

df_encoded = pd.concat([df_encoded.drop(columns = ['LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith']), 
                        language_dummies, database_dummies], axis = 1)

In [33]:
df_encoded = df_encoded.drop(columns = ['Age', 'EdLevel', 'OrgSize'])

## ENCODING DONE!

## Train/Test & Modelling: Linear Regression

In [34]:
# training on one portion of data, testing on unseen data to evaluate model's effectiveness at generalizing

from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns = ['annual_salary_usd', 'ResponseId'])
y = df_encoded['annual_salary_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 3542, Test: 886


In [35]:
# fitting a baseline model, going with linear regression
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

## Evaluating R^2 and some stats

In [36]:
# evaluating the model to see if any changes should be made
# changes will have to be made, prob a different model might work
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

linear_r2 = r2_score(y_test, y_pred)
linear_mae = mean_absolute_error(y_test, y_pred)
linear_rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f'R2: {linear_r2:.3f}')
print(f'MAE: ${linear_mae:.0f}')
print(f'RMSE: ${linear_rmse:.0f}')

R2: 0.533
MAE: $30985
RMSE: $46641


## Linear Regression Model Summary:
The first model chosen, linear regression, was chosen based on my knowledge that it is basic, simplistic, and could pick up any obvious patterns. The model achieved an R^2 of 0.533, meaning that 53.3% of the variation in Y is explained by X. Putting the Mean Absolute Error in USD was extremely helpful, indicating that on average, the baseline model's salary was about $31.0K away from the actual salary. Definitely some modest error, but I chose Random Forest Regression next because it might improve the output slightly. Random Forest Models can pick up nonlinear relationships and interactions between features that Linear Regression may miss.

## Train/Test & Modeling: Random Forest Regression

In [37]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42,
    n_jobs = 1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [38]:
# evaluating new, Random Forest Model
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = mean_squared_error(y_test, rf_pred) ** 0.5

print(f'Random Forest R2: {rf_r2:.3f}')
print(f'Random Forest MAE: ${rf_mae:.0f}')
print(f'Random Forest RMSE: ${rf_rmse:.0f}')

Random Forest R2: 0.539
Random Forest MAE: $31481
Random Forest RMSE: $46337


## Model Comparisons

| Model | R^2 | MAE | RMSE |
|---|---|---|---|
| Linear Regression  | 0.533 | $30,985 | $46,641 |
| Random Forest | 0.539 | $31,481 | $46,337 |

**Summary:** The Random Forest Model performed slightly better in terms of R^2 and RMSE, while the Linear Model performed slightly better with a lower MAE. Overall, the models performed similarly and neither substantially outperformed the other. Random Forest explained 53.3% in variation while the Linear explained 53.9%. However, its average absolute error was higher by about $500. 

## Error Analytics

In [39]:
# comparing prediction errors
linear_errors = y_test - y_pred
rf_errors = y_test - rf_pred

print(f'Linear Regression mean error: ${linear_errors.mean():,.0f}')
print(f'Random Forest mean error: ${rf_errors.mean():,.0f}')

Linear Regression mean error: $1,494
Random Forest mean error: $-458


The mean error for Linear Regression is positive at $1,494. This indicates that it tends to UNDERpredict salary. On the other hand, the mean error for Random Forest is negative at -458 dollars, indicating that it OVERpredicts salary. 

## Limitations & Future Improvements

The model explains about 53% of the variation in salary, so a substantial amount of variation still remains unexplained. This could be due to factors the survey did not capture, like specific employer, seniority, location within a country, or other individual factors.

With a little more time, I would explore deeper into cross validating features and hyperparameter tuning for the model, and then turn my attention to deeper feature engineering and potentially other models such as gradient boosting. I would also investigate whether the largest prediction errors are concentrated among particular groups of respondants.

In [40]:
print(X.columns.tolist())

['WorkExp', 'YearsCode', 'Age_encoded', 'EdLevel_encoded', 'OrgSize_encoded', 'DevType_Academic researcher', 'DevType_Applied scientist', 'DevType_Architect, software or solutions', 'DevType_Cloud infrastructure engineer', 'DevType_Cybersecurity or InfoSec professional', 'DevType_Data engineer', 'DevType_Data or business analyst', 'DevType_Data scientist', 'DevType_Database administrator or engineer', 'DevType_DevOps engineer or professional', 'DevType_Developer, AI apps or physical AI', 'DevType_Developer, QA or test', 'DevType_Developer, back-end', 'DevType_Developer, desktop or enterprise applications', 'DevType_Developer, embedded applications or devices', 'DevType_Developer, front-end', 'DevType_Developer, full-stack', 'DevType_Developer, game or graphics', 'DevType_Developer, mobile', 'DevType_Engineering manager', 'DevType_Financial analyst or engineer', 'DevType_Founder, technology or otherwise', 'DevType_Other (please specify):', 'DevType_Product manager', 'DevType_Project man